In [1]:
import pandas as pd


In [ ]:
df = pd.read_csv('/home/jingjie/AutoTorch/src/eval/idfraud/annotations_with_absolute_paths.csv')

df = df[df['dataset'] != 'myid_20260222_10PM_cron_id_fraud_reject'] # drop wrong dataset 
df = df.drop(index=[525, 3163]) # drop unlabeled

df[df.isna().any(axis=1)]  

df['tag'] = (
    df['tag']
    .str.strip()              # remove leading/trailing spaces
    .str.replace(r'\s+', ' ', regex=True)  # replace multiple spaces with one
)
df['tag'] = df['tag'].str.split('/').apply(lambda x: [i.strip() for i in x])


In [3]:
has_quality  = df['tag'].apply(lambda x: 'Quality' in x)
has_unknown  = df['tag'].apply(lambda x: 'Unknown' in x)
quality_only = df['tag'].apply(lambda x: set(x) == {'Quality'})

# exclude any row where Quality or Unknown appears anywhere
df_exclude_unknown_and_quality = df[~(has_quality | has_unknown)]

# exclude Quality-only and Unknown (keep Genuine/Quality etc.)
df_exclude_quality_only_and_unknown = df[~(quality_only | has_unknown)]

for label, frame, mask in [
    ('exclude all Quality & Unknown',   df_exclude_unknown_and_quality,      has_quality | has_unknown),
    ('exclude Quality-only & Unknown',  df_exclude_quality_only_and_unknown,  quality_only | has_unknown),
]:
    excluded = df[mask]
    print(f"=== {label} ===")
    print(f"Before : {len(df):,}")
    print(f"Removed: {len(excluded):,}")
    print(f"  - Quality (any)  : {has_quality[mask].sum():,}")
    print(f"  - Quality-only   : {quality_only[mask].sum():,}")
    print(f"  - Unknown        : {has_unknown[mask].sum():,}")
    print(f"After  : {len(frame):,}")
    print(f"Remaining tags: {sorted(frame['tag'].apply(tuple).unique())}")
    print()

=== exclude all Quality & Unknown ===
Before : 5,596
Removed: 2,269
  - Quality (any)  : 2,266
  - Quality-only   : 5
  - Unknown        : 3
After  : 3,327
Remaining tags: [('Cutout Printed - Color',), ('Cutout Printed - Grayscale',), ('Genuine',), ('Genuine', 'Replay - Mobile'), ('Genuine', 'Tamper - Face'), ('Printed - Color',), ('Printed - Grayscale',), ('Replay - Mobile',), ('Replay - Monitor',), ('Replay - Tablet',), ('Tamper - Face',), ('Tamper - Photo',)]

=== exclude Quality-only & Unknown ===
Before : 5,596
Removed: 8
  - Quality (any)  : 5
  - Quality-only   : 5
  - Unknown        : 3
After  : 5,588
Remaining tags: [('Cutout Printed - Color',), ('Cutout Printed - Grayscale',), ('Genuine',), ('Genuine', 'Quality'), ('Genuine', 'Replay - Mobile'), ('Genuine', 'Tamper - Face'), ('Printed - Color',), ('Printed - Grayscale',), ('Quality', 'Genuine'), ('Replay - Mobile',), ('Replay - Monitor',), ('Replay - Tablet',), ('Tamper - Face',), ('Tamper - Photo',)]



In [4]:
df_exclude_unknown_and_quality['label'] = df['tag'].apply(lambda x: 0 if 'Genuine' in str(x) else 1)
df_exclude_quality_only_and_unknown['label'] = df['tag'].apply(lambda x: 0 if 'Genuine' in str(x) else 1)

print(f"Label distribution:")
print(df_exclude_unknown_and_quality['label'].value_counts())
print(df_exclude_quality_only_and_unknown['label'].value_counts())


Label distribution:
label
0    2825
1     502
Name: count, dtype: int64
label
0    5086
1     502
Name: count, dtype: int64


/tmp/ipykernel_2280250/824594608.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_exclude_unknown_and_quality['label'] = df['tag'].apply(lambda x: 0 if 'Genuine' in str(x) else 1)
/tmp/ipykernel_2280250/824594608.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_exclude_quality_only_and_unknown['label'] = df['tag'].apply(lambda x: 0 if 'Genuine' in str(x) else 1)


In [5]:
import sys
import os
import torch
import yaml
from tqdm import tqdm
from torch.utils.data import DataLoader

sys.path.insert(0, '/home/jingjie/AutoTorch/src')

from data.idfraud.dataset import IDFraudTorchDataset
from data.idfraud.transforms import get_transform
from models import build_classifier_model, load_weights_from_checkpoint
from models.dino import load_dino_model

In [6]:
# Config paths
ORI_EXP = "/home/jingjie/AutoTorch/runs/Exp2_dinov3_convnext_large_v1_512_ori"
CROP_EXP = "/home/jingjie/AutoTorch/runs/Exp2_dinov3_convnext_large_v1_512_crop"
BEST_CKPT_ORI = 1   # checkpoint for ORI model
BEST_CKPT_CROP = 10  # checkpoint for CROP model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load config (same for both since same model architecture)
with open(os.path.join(ORI_EXP, "config.yaml")) as f:
    cfg = yaml.safe_load(f)

# Override head_type to match checkpoint (was trained with v1)
cfg['model']['head_type'] = 'v1'

transform = get_transform(cfg)

# Load backbone once (shared)
print("Loading backbone...")
backbone, backbone_dim = load_dino_model(cfg)

Loading backbone...


/home/jingjie/.conda/envs/vision/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
def run_inference(df, path_col, exp_dir, ckpt_num, backbone, backbone_dim, cfg, transform, device, batch_size=32):
    """Run inference using specified path column and checkpoint."""
    # Prepare df with 'path' column
    df_temp = df.copy()
    df_temp['path'] = df_temp[path_col]
    
    # Create dataset and dataloader
    dataset = IDFraudTorchDataset(df_temp, transform=transform)
    dataloader = DataLoader(dataset, batch_size=batch_size, num_workers=8, pin_memory=True)
    
    # Load model
    ckpt_path = os.path.join(exp_dir, "checkpoints", f"epoch_{ckpt_num}.pt")
    model = build_classifier_model(cfg, device, backbone, backbone_dim)
    model = load_weights_from_checkpoint(model, ckpt_path, device)
    model.eval()
    
    # Run inference
    all_probs = []
    with torch.inference_mode():
        for X, _ in tqdm(dataloader, desc=f"Inference {os.path.basename(exp_dir)}"):
            logits = model(X.to(device)).squeeze(1)
            all_probs.extend(torch.sigmoid(logits).cpu().tolist())
    
    return all_probs

In [8]:
# Run predictions with ORI model (using absolute_ori_path)
print("Running ORI model predictions...")

import os
df_exclude_unknown_and_quality = df_exclude_unknown_and_quality[df_exclude_unknown_and_quality['absolute_ori_path'].apply(os.path.exists) & df_exclude_unknown_and_quality['absolute_ocr_path'].apply(os.path.exists)]

print(f"Label distribution:")
print(df_exclude_unknown_and_quality['label'].value_counts())

                                          
df_exclude_unknown_and_quality['pred_ori'] = run_inference(df_exclude_unknown_and_quality, 'absolute_ori_path', ORI_EXP, BEST_CKPT_ORI, backbone, backbone_dim, cfg, transform, device)

# Run predictions with CROP model (using absolute_ocr_path as crop)
print("\nRunning CROP model predictions...")
df_exclude_unknown_and_quality['pred_crop'] = run_inference(df_exclude_unknown_and_quality, 'absolute_ocr_path', CROP_EXP, BEST_CKPT_CROP, backbone, backbone_dim, cfg, transform, device)

Running ORI model predictions...
Label distribution:
label
0    2764
1     494
Name: count, dtype: int64


Inference Exp2_dinov3_convnext_large_v1_512_ori: 100%|██████████| 102/102 [00:45<00:00,  2.26it/s]



Running CROP model predictions...


Inference Exp2_dinov3_convnext_large_v1_512_crop: 100%|██████████| 102/102 [00:42<00:00,  2.38it/s]


In [9]:
# Combine predictions: average ori and crop, threshold at 0.5
df_exclude_unknown_and_quality['avg_pred'] = (df_exclude_unknown_and_quality['pred_ori'] + df_exclude_unknown_and_quality['pred_crop']) / 2
df_exclude_unknown_and_quality['final_pred'] = (df_exclude_unknown_and_quality['avg_pred'] > 0.5).astype(int)

print(f"Predictions complete!")
print(f"Total samples: {len(df_exclude_unknown_and_quality)}")
print(f"\nPrediction distribution:")
print(df_exclude_unknown_and_quality['final_pred'].value_counts())

# Show sample results
df_exclude_unknown_and_quality[['dataset', 'label', 'pred_ori', 'pred_crop', 'avg_pred', 'final_pred']].head(10)
df_exclude_unknown_and_quality.to_csv("temp.csv")

Predictions complete!
Total samples: 3258

Prediction distribution:
final_pred
0    2638
1     620
Name: count, dtype: int64


In [12]:
# Per-dataset metrics
def calc_metrics_series(grp):
    attacks = grp[grp['label'] == 1]
    genuines = grp[grp['label'] == 0]
    
    apcer = (attacks['final_pred'] == 0).sum() / len(attacks) if len(attacks) > 0 else None
    bpcer = (genuines['final_pred'] == 1).sum() / len(genuines) if len(genuines) > 0 else None
    accuracy = (grp['label'] == grp['final_pred']).mean()
    
    return pd.Series({
        'APCER': apcer,
        'BPCER': bpcer,
        'Accuracy': accuracy,
        'n_attacks': len(attacks),
        'n_genuines': len(genuines)
    })

dataset_metrics = df_exclude_unknown_and_quality.groupby('dataset').apply(calc_metrics_series).reset_index()
pd.set_option('display.max_rows', None)
dataset_metrics

/tmp/ipykernel_2280250/2818197124.py:18: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  dataset_metrics = df_exclude_unknown_and_quality.groupby('dataset').apply(calc_metrics_series).reset_index()


,dataset,APCER,BPCER,Accuracy,n_attacks,n_genuines
0,myid_20260215_10AM_cron_idfraud_accept,NaN,0.000000,1.000000,0.0,25.0
1,myid_20260215_10AM_cron_idfraud_reject,0.000000,0.055556,0.954545,4.0,18.0
2,myid_20260215_10PM_cron_idfraud_accept,NaN,0.000000,1.000000,0.0,21.0
3,myid_20260215_10PM_cron_idfraud_reject,0.000000,0.181818,0.866667,4.0,11.0
4,myid_20260216_10AM_cron_idfraud_accept,NaN,0.000000,1.000000,0.0,32.0
5,myid_20260216_10AM_cron_idfraud_reject,0.000000,0.200000,0.862069,9.0,20.0
6,myid_20260216_10PM_cron_idfraud_accept,NaN,0.000000,1.000000,0.0,35.0
7,myid_20260216_10PM_cron_idfraud_reject,0.000000,0.153846,0.857143,2.0,26.0
8,myid_20260217_10AM_cron_idfraud_accept,NaN,0.000000,1.000000,0.0,30.0
9,myid_20260217_10AM_cron_idfraud_reject,0.400000,0.074074,0.875000,5.0,27.0


In [ ]:
# Threshold search
import numpy as np

print("Thresh\tAPCER\tBPCER\tAccuracy")
print("-" * 40)

for thresh in np.arange(0.05, 0.95, 0.025):
    pred = (df_exclude_unknown_and_quality['avg_pred'] > thresh).astype(int)
    
    attacks = df_exclude_unknown_and_quality[df_exclude_unknown_and_quality['label'] == 1]
    genuines = df_exclude_unknown_and_quality[df_exclude_unknown_and_quality['label'] == 0]
    
    apcer = (attacks['avg_pred'] <= thresh).sum() / len(attacks)
    bpcer = (genuines['avg_pred'] > thresh).sum() / len(genuines)
    accuracy = (pred == df_exclude_quality_only_and_unknown['label']).mean()
    
    print(f"{thresh:.2f}\t{apcer:.4f}\t{bpcer:.4f}\t{accuracy:.4f}")

In [13]:
# APCER: attack samples missed / total attacks
# BPCER: genuine samples falsely flagged / total genuines
attacks = df_exclude_unknown_and_quality[df_exclude_unknown_and_quality['label'] == 1]
genuines = df_exclude_unknown_and_quality[df_exclude_unknown_and_quality['label'] == 0]

apcer = (attacks['final_pred'] == 0).sum() / len(attacks)
bpcer = (genuines['final_pred'] == 1).sum() / len(genuines)

print(f"APCER: {apcer:.4f}")
print(f"BPCER: {bpcer:.4f}")


APCER: 0.0951
BPCER: 0.0626
